# 11. Contextual Chunk Headers

Prepend a descriptive header (document + section) to every chunk before embedding.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

A chunk lifted out of its document often loses its subject: *"It is usually measured with cosine similarity"* - what is *it*? **Contextual chunk headers** prepend a short description (document title, section name, or an LLM-written summary line) to each chunk **before embedding**, so the vector knows what the chunk is about.

## How it works

1. Split the document by structure (sections)
2. Build a header per chunk: `Document > Section`
3. Prepend the header to the chunk text
4. Embed the header + chunk together
5. At answer time you can pass the header too, so the LLM also sees the context

```text
 raw chunk:      "It is usually measured with cosine similarity..."
 with header:    "The Applied AI Handbook > Embeddings
                  It is usually measured with cosine similarity..."  --> embed
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *

sections = split_sections(load_text())
chunks, headers = [], []
for heading, body in sections:
    for c in chunk_text(body, size=350, overlap=40):
        chunks.append(c)
        headers.append(f"The Applied AI Handbook > {heading}")
print(len(chunks), "chunks")

A chunk like *"It is usually measured with cosine similarity"* is ambiguous alone. The header tells the embedder what **it** is.

In [ ]:
plain = build_store(chunks)
with_headers = build_store([f"{h}\n{c}" for h, c in zip(headers, chunks)])

query = "How is similarity measured for embeddings?"
print("--- plain"); show(plain.search(query, 3))
print("--- with headers"); show(with_headers.search(query, 3))

In [ ]:
hits = with_headers.search(query, 3)
print(answer(query, [h["text"] for h in hits]))

Headers can also be LLM-generated when documents have no clear structure: `llm(f"Write a short title for this passage: {chunk}")`.

## Evaluate: plain vs headers
Chunks come from section-aware splitting, so the same text is indexed with and without headers.

In [ ]:
compare_retrievers({"plain chunks": plain.search, "chunks + header": with_headers.search}, k=3)

## LLM-generated headers
When documents have no headings, ask the LLM to write one line of context for each chunk.

In [ ]:
def llm_header(chunk):
    return llm(f"Write a short descriptive title (max 10 words) for this passage. Reply with the title only.\n\n{chunk}")

for c in chunks[:4]:
    print(llm_header(c), "<-", c[:70].replace("\n", " "))

In [ ]:
llm_headers = [llm_header(c) for c in chunks]
llm_store = build_store([f"{h}\n{c}" for h, c in zip(llm_headers, chunks)])
compare_retrievers({"plain": plain.search, "static headers": with_headers.search, "LLM headers": llm_store.search}, k=3)

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Cheap and easy | Needs structure or an LLM pass to make headers |
| Disambiguates pronoun-heavy chunks | Long headers dilute the chunk's own content |
| Works with any retriever | Gains small on already self-contained chunks |

## When to use it

Structured documents (manuals, reports, wikis, contracts) where chunks depend on their section.

## Experiments to try

- Add a header level (`Document > Chapter > Section`).
- Include the previous chunk's last sentence in the header.
- Compare header length 5 vs 20 words.

## Key takeaways

- Context belongs in the embedded text.
- Structure you already have is the cheapest header.
- Evaluate - gains depend on how ambiguous your chunks are.

---
**Next:** [12. Context Enrichment - Window Around Chunk](../12_context_enrichment_window_around_chunk/context_enrichment_window_around_chunk.ipynb)